*0.1 Python for GenAI*

# Generators

**The situation.** Your team exports the year's support tickets to search them: one file, 50 GB, one ticket per line. The first script loads the file with `json.load`, which builds the whole thing in memory. The machine has 16 GB. The script dies at 30%. On a bigger machine it works — after 20 minutes of loading during which nothing else happens.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The fix: read one line at a time and forget it.** A *generator* is a function that hands out one item with `yield`, pauses, and continues when the next item is asked for. Chain a few of them — read, filter, transform, write — and only one ticket is ever in memory.

**First, make a realistic file.** 100,000 tickets, one JSON object per line.

In [2]:
import json
import tempfile
from pathlib import Path

folder = tempfile.mkdtemp()
path = Path(folder) / "tickets.jsonl"
with path.open("w") as handle:
    for number in range(100_000):
        status = "open" if number % 3 == 0 else "closed"
        ticket = {
            "id": number,
            "status": status,
            "text": "customer cannot log in " * (number % 20 + 1),
        }
        handle.write(json.dumps(ticket) + "\n")
print("file size:", round(path.stat().st_size / 1e6, 1), "MB")

file size: 28.7 MB


**Now the pipeline.** Three small generators. Notice that calling them does nothing yet — no line is read until the final loop asks for one.

In [3]:
from collections.abc import Iterator


def read_tickets(path: Path) -> Iterator[dict]:
    with path.open() as handle:
        for line in handle:
            yield json.loads(line)  # hand out one ticket, then pause until asked for the next


def only_open(tickets: Iterator[dict]) -> Iterator[dict]:
    for ticket in tickets:
        if ticket["status"] == "open":
            yield ticket


def with_word_count(tickets: Iterator[dict]) -> Iterator[dict]:
    for ticket in tickets:
        ticket["words"] = len(ticket["text"].split())
        yield ticket


pipeline = with_word_count(only_open(read_tickets(path)))  # nothing has been read yet
print("pipeline built:", type(pipeline).__name__)

processed = 0
total_words = 0
for ticket in pipeline:  # each turn pulls one ticket through all three stages
    processed += 1
    total_words += ticket["words"]
print("open tickets processed:", processed, "| words counted:", total_words)
assert processed == 33_334

pipeline built: generator
open tickets processed: 33334 | words counted: 1400028


**Reading the output.** 33,334 open tickets (every third one) flowed through read → filter → count, one at a time. The file could be 50 GB or 500 GB; the memory used is the same — one ticket.

```
50 GB file ──▶ read one line ──▶ keep if open ──▶ count words ──▶ next line
                    ▲                                                │
                    └────────────────── one ticket in memory ────────┘
```

**The rule to remember.** When data is bigger than memory, or when the first result should appear right away, build the job as a chain of generators.

| Use it when | Don't when | Instead use |
|---|---|---|
| files bigger than memory; ETL and ingestion jobs; log processing | you need to go over the data twice, or jump to item 40,000 | a list or a database table |

**Watch out**
- A generator can be read once. If a later step needs the data again, write it out (a file, a table).
- `list(pipeline)` — "just to count them" — loads everything back into memory. The classic mistake.
- Record progress (the line number) every few thousand tickets, so a crash at ticket 40,000,000 resumes instead of restarting.